In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_767.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/skeleton_utils.py
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_1757.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_815.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_71.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/frame_sampling.py
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_1710.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_209.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_478.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_1730.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Absolute_Data.csv
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_747.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_1797.0.mp4
/kaggle/input/datasets/arnobsaha1/weightgait-pilot/Instance_229.0.mp4
/kaggle/input/datase

In [2]:
!pip install -q -U transformers accelerate qwen-vl-utils opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.4 MB/s eta 0:00:00:00:0100:01


In [3]:
import sys
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')

from frame_sampling import sample_frame_indices, extract_frames
from kaggle_qwen_backend import generate_fn
from harness import classify_frames_local
from labels import load_labels, get_instance_label
from skeleton_utils import load_instance_joints, active_frame_range

In [4]:
import cv2

DATA_DIR = '/kaggle/input/datasets/arnobsaha1/weightgait-pilot'
CSV_PATH = f'{DATA_DIR}/Absolute_Data.csv'
VIDEO_PATH = f'{DATA_DIR}/Instance_785.0.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

instance_joints = load_instance_joints(CSV_PATH, 785)
start, end = active_frame_range(instance_joints)
indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
frames = extract_frames(VIDEO_PATH, indices)

prompt = (
    'You are an expert in clinical gait analysis. You are shown a sequence of frames '
    'sampled evenly across a video of a person walking. Carefully examine their '
    'walking movement across the frames, paying attention to: step length and '
    'symmetry between the left and right leg, foot clearance from the ground during '
    'each step, arm swing pattern, stride timing, and any dragging, hesitation, or '
    'uneven weight distribution between the legs. '
    'Base your judgment on the walking movement pattern itself, not on clothing, '
    'footwear, or other visual details unrelated to how the person moves. '
    'Classify their gait into exactly one of the following three categories: '
    '0 = normal walking, 1 = limping (one leg affected), 2 = shuffling (both legs '
    'affected). Respond only with valid JSON in this exact format: '
    '{"weight_class": <0, 1, or 2>, "reasoning": "<2-3 sentences describing the '
    'specific gait characteristics you observed that led to this classification>"}'
)

result = classify_frames_local(generate_fn, frames, prompt)
print(result)

labels = load_labels(CSV_PATH)
print("Ground truth:", get_instance_label(labels, 785))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

{'video_path': None, 'raw_response': '{"weight_class": 0, "reasoning": "The person exhibits a symmetrical gait with consistent step length and regular stride timing. There is no visible dragging, hesitation, or uneven weight distribution; both legs clear the ground evenly and arms swing naturally in coordination with the steps."}', 'parsed': {'weight_class': 0, 'reasoning': 'The person exhibits a symmetrical gait with consistent step length and regular stride timing. There is no visible dragging, hesitation, or uneven weight distribution; both legs clear the ground evenly and arms swing naturally in coordination with the steps.'}, 'success': True, 'error': None}
Ground truth: {'weight_class': 0, 'gait_freeze': False, 'obstacle': False, 'person_id': 13}


In [5]:
DEV_INSTANCES = [71, 189, 438, 494, 727, 785, 795, 851, 1690, 1757,
                  91, 209, 458, 514, 747, 805, 815, 871, 1710, 1777,
                  111, 229, 478, 534, 767, 825, 835, 891, 1730, 1797]

results = []
for instance in DEV_INSTANCES:
    video_path = f'{DATA_DIR}/Instance_{instance}.0.mp4'

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if total_frames == 0:
        print(f"[{instance}] SKIPPED -- video not found or unreadable at {video_path}")
        continue

    instance_joints = load_instance_joints(CSV_PATH, instance)
    start, end = active_frame_range(instance_joints)
    indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
    frames = extract_frames(video_path, indices)

    result = classify_frames_local(generate_fn, frames, prompt)
    ground_truth = get_instance_label(labels, instance)

    predicted = result["parsed"]["weight_class"] if result["success"] else None
    correct = predicted == ground_truth["weight_class"] if result["success"] else None
    results.append({"instance": instance, "predicted": predicted, "ground_truth": ground_truth["weight_class"], "correct": correct})

    if result["success"]:
        mark = "OK" if correct else "WRONG"
        print(f"[{instance}] gt={ground_truth['weight_class']} pred={predicted} ({mark}) -- {result['parsed']['reasoning']}")
    else:
        print(f"[{instance}] FAILED -- {result['error']}")

n_scored = sum(1 for r in results if r["correct"] is not None)
n_correct = sum(1 for r in results if r["correct"])
print(f"\nAccuracy: {n_correct}/{n_scored}" + (f" = {n_correct/n_scored:.2f}" if n_scored else ""))

[71] gt=0 pred=0 (OK) -- The individual displays a symmetrical gait with consistent step length and normal foot clearance. Arms swing naturally in coordination with leg movement, and there is no visible dragging, hesitation, or uneven weight distribution, indicating normal walking.
[189] gt=0 pred=0 (OK) -- The individual displays a symmetrical, rhythmic gait with consistent step length and proper foot clearance during each stride. There is no visible dragging, hesitation, or uneven weight distribution, indicating normal walking mechanics.
[438] gt=0 pred=0 (OK) -- The gait shows normal symmetry with alternating leg steps, consistent foot clearance, and coordinated arm swing. There is no evidence of dragging, hesitation, or uneven weight distribution, indicating a typical, balanced walking pattern.
[494] gt=0 pred=0 (OK) -- The individual displays a symmetrical gait with consistent step length and regular stride timing. There is no visible dragging, hesitation, or uneven weight distrib

In [6]:
REPRESENTATIONS = ['raw_video', 'skeleton', 'overlay', 'montage']
DEV_INSTANCES = [71, 189, 438, 494, 727, 785, 795, 851, 1690, 1757,
                  91, 209, 458, 514, 747, 805, 815, 871, 1710, 1777,
                  111, 229, 478, 534, 767, 825, 835, 891, 1730, 1797]

def get_frames_for_representation(instance, representation):
    if representation == 'raw_video':
        video_path = f'{DATA_DIR}/Instance_{instance}.0.mp4'
    elif representation in ('skeleton', 'overlay'):
        video_path = f'{DATA_DIR}/kaggle_representations/{instance}/{representation}.mp4'
    elif representation == 'montage':
        path = f'{DATA_DIR}/kaggle_representations/{instance}/montage.png'
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Could not read {path}")
        return [img]
    else:
        raise ValueError(representation)

    instance_joints = load_instance_joints(CSV_PATH, instance)
    start, end = active_frame_range(instance_joints)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
    frames = extract_frames(video_path, indices)
    if not frames:
        raise FileNotFoundError(f"No frames read from {video_path} -- check the path")
    return frames

results = []
for instance in DEV_INSTANCES:
    ground_truth = get_instance_label(labels, instance)
    for representation in REPRESENTATIONS:
        try:
            frames = get_frames_for_representation(instance, representation)
        except FileNotFoundError as e:
            print(f"[{instance}][{representation}] SKIPPED -- {e}")
            continue

        result = classify_frames_local(generate_fn, frames, prompt)
        predicted = result['parsed']['weight_class'] if result['success'] else None
        correct = predicted == ground_truth['weight_class'] if predicted is not None else None
        results.append({'instance': instance, 'representation': representation, 'predicted': predicted, 'correct': correct})
        mark = 'OK' if correct else ('WRONG' if correct is not None else 'FAILED')
        print(f"[{instance}][{representation}] gt={ground_truth['weight_class']} pred={predicted} ({mark})")

print("\nAccuracy by representation:")
for representation in REPRESENTATIONS:
    scored = [r for r in results if r['representation'] == representation and r['correct'] is not None]
    n_correct = sum(r['correct'] for r in scored)
    print(f"  {representation}: {n_correct}/{len(scored)}" + (f" = {n_correct/len(scored):.2f}" if scored else ""))

[71][raw_video] gt=0 pred=0 (OK)
[71][skeleton] gt=0 pred=1 (WRONG)
[71][overlay] gt=0 pred=0 (OK)
[71][montage] gt=0 pred=0 (OK)
[189][raw_video] gt=0 pred=0 (OK)
[189][skeleton] gt=0 pred=1 (WRONG)
[189][overlay] gt=0 pred=0 (OK)
[189][montage] gt=0 pred=0 (OK)
[438][raw_video] gt=0 pred=0 (OK)
[438][skeleton] gt=0 pred=0 (OK)
[438][overlay] gt=0 pred=0 (OK)
[438][montage] gt=0 pred=0 (OK)
[494][raw_video] gt=0 pred=0 (OK)
[494][skeleton] gt=0 pred=1 (WRONG)
[494][overlay] gt=0 pred=0 (OK)
[494][montage] gt=0 pred=0 (OK)
[727][raw_video] gt=0 pred=0 (OK)
[727][skeleton] gt=0 pred=0 (OK)
[727][overlay] gt=0 pred=0 (OK)
[727][montage] gt=0 pred=0 (OK)
[785][raw_video] gt=0 pred=0 (OK)
[785][skeleton] gt=0 pred=0 (OK)
[785][overlay] gt=0 pred=0 (OK)
[785][montage] gt=0 pred=0 (OK)
[795][raw_video] gt=0 pred=0 (OK)
[795][skeleton] gt=0 pred=1 (WRONG)
[795][overlay] gt=0 pred=0 (OK)
[795][montage] gt=0 pred=0 (OK)
[851][raw_video] gt=0 pred=0 (OK)
[851][skeleton] gt=0 pred=0 (OK)
[851][ov

In [11]:
import cv2

DATA_DIR = '/kaggle/input/datasets/arnobsaha1/weightgait-pilot'
CSV_PATH = f'{DATA_DIR}/Absolute_Data.csv'
VIDEO_PATH = f'{DATA_DIR}/Instance_785.0.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

instance_joints = load_instance_joints(CSV_PATH, 785)
start, end = active_frame_range(instance_joints)
indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
frames = extract_frames(VIDEO_PATH, indices)

prompt = (
    'You are an expert in clinical gait analysis. You are shown a sequence of frames '
    'sampled evenly across a video of a person walking. Carefully examine their '
    'walking movement across the frames, paying attention to: step length and '
    'symmetry between the left and right leg, foot clearance from the ground during '
    'each step, arm swing pattern, stride timing, and any dragging, hesitation, or '
    'uneven weight distribution between the legs. '
    'Base your judgment on the walking movement pattern itself, not on clothing, '
    'footwear, or other visual details unrelated to how the person moves. '
    'Classify their gait into exactly one of the following three categories: '
    '0 = normal walking, 1 = limping (one leg affected), 2 = shuffling (both legs '
    'affected). Respond only with valid JSON in this exact format: '
    '{"weight_class": <0, 1, or 2>, "reasoning": "<2-3 sentences describing the '
    'specific gait characteristics you observed that led to this classification>"}'
)

result = classify_frames_local(generate_fn, frames, prompt)
print(result)

labels = load_labels(CSV_PATH)
print("Ground truth:", get_instance_label(labels, 785))

{'video_path': None, 'raw_response': '{\n  "weight_class": 0,\n  "reasoning": "The person demonstrates a symmetrical gait with consistent step length and regular stride timing. Both legs clear the ground evenly, and the arm swing is balanced, indicating normal walking without signs of limping or shuffling."\n}', 'parsed': {'weight_class': 0, 'reasoning': 'The person demonstrates a symmetrical gait with consistent step length and regular stride timing. Both legs clear the ground evenly, and the arm swing is balanced, indicating normal walking without signs of limping or shuffling.'}, 'success': True, 'error': None}
Ground truth: {'weight_class': 0, 'gait_freeze': False, 'obstacle': False, 'person_id': 13}


In [12]:
from harness import parse_model_response

def classify_with_temp(frames, prompt, temperature=None):
    raw_text = generate_fn(frames, prompt, temperature=temperature)
    parsed, success, error = parse_model_response(raw_text)
    return {"video_path": None, "raw_response": raw_text, "parsed": parsed, "success": success, "error": error}

print("\n=== Self-consistency: same blind prompt x5 at temperature=0.8 ===")
blind_repeats = []
for i in range(5):
    r = classify_with_temp([], prompt, temperature=0.8)
    blind_repeats.append(r['parsed']['weight_class'] if r['success'] else r.get('error'))
    print(f"  run {i+1}: {r['parsed'] if r['success'] else r['error']}")
print("blind predictions x5:", blind_repeats)

print(f"\n=== Skeleton consistency check, instance {test_instance} (gt=2), 5 repeats ===")
skel_repeats = []
for i in range(5):
    r = classify_with_temp(frames, prompt, temperature=0.8)
    skel_repeats.append(r['parsed']['weight_class'] if r['success'] else r.get('error'))
    print(f"  run {i+1}: {r['parsed'] if r['success'] else r['error']}")
print(f"instance {test_instance} skeleton predictions x5:", skel_repeats)


=== Self-consistency: same blind prompt x5 at temperature=0.8 ===
  run 1: {'weight_class': 0, 'reasoning': 'The gait shows symmetrical step length, consistent foot clearance, and balanced arm swing with regular stride timing. There is no visible dragging, hesitation, or uneven weight distribution, indicating normal walking mechanics.'}
  run 2: {'weight_class': 0, 'reasoning': 'The gait exhibits symmetrical step length, consistent foot clearance, and balanced arm swing with regular stride timing. There is no visible dragging, hesitation, or uneven weight distribution, indicating normal walking mechanics.'}
  run 3: {'weight_class': 0, 'reasoning': 'The gait exhibits symmetrical step length, consistent foot clearance, balanced arm swing, and regular stride timing with no visible dragging, hesitation, or weight asymmetry, indicating normal walking.'}
  run 4: {'weight_class': 0, 'reasoning': 'The gait shows symmetrical step length, consistent foot clearance, balanced arm swing, and reg

In [13]:
test_instance_2 = 91  # gt=1, skeleton predicted 1 correctly on the greedy run
video_path_2 = f'{DATA_DIR}/kaggle_representations/{test_instance_2}/skeleton.mp4'
instance_joints_2 = load_instance_joints(CSV_PATH, test_instance_2)
start2, end2 = active_frame_range(instance_joints_2)
cap = cv2.VideoCapture(video_path_2)
total_frames_2 = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
indices_2 = sample_frame_indices(total_frames_2, n=8, start=start2, end=end2)
frames_2 = extract_frames(video_path_2, indices_2)

print(f"=== Skeleton consistency check, instance {test_instance_2} (gt=1), 5 repeats ===")
skel_repeats_2 = []
for i in range(5):
    r = classify_with_temp(frames_2, prompt, temperature=0.8)
    skel_repeats_2.append(r['parsed']['weight_class'] if r['success'] else r.get('error'))
    print(f"  run {i+1}: {r['parsed'] if r['success'] else r['error']}")
print(f"instance {test_instance_2} skeleton predictions x5:", skel_repeats_2)

=== Skeleton consistency check, instance 91 (gt=1), 5 repeats ===
  run 1: {'weight_class': 1, 'reasoning': 'The gait shows a clear asymmetry with the right leg taking a longer step and the left leg appearing shorter and more hesitant, suggesting a limp. The left foot does not clear the ground as effectively, indicating impaired weight-bearing on that side.'}
  run 2: {'weight_class': 1, 'reasoning': 'The gait shows an asymmetrical pattern with the right leg taking longer, more pronounced steps while the left leg appears shorter and less coordinated, suggesting a limp. There is also a noticeable difference in foot clearance and stride timing between the legs, consistent with unilateral gait deviation.'}
  run 3: {'weight_class': 1, 'reasoning': 'The gait shows a clear asymmetry with the right leg taking longer, more powerful steps while the left leg is consistently shorter and appears to be compensating, indicating a limp. The left foot clearance is reduced, suggesting impaired functio

In [14]:
for test_id in [209, 815, 1710]:
    vp = f'{DATA_DIR}/kaggle_representations/{test_id}/skeleton.mp4'
    ij = load_instance_joints(CSV_PATH, test_id)
    s, e = active_frame_range(ij)
    cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    idx = sample_frame_indices(tf, n=8, start=s, end=e)
    fr = extract_frames(vp, idx)
    r = classify_frames_local(generate_fn, fr, prompt)
    print(f"[{test_id}] gt=1 pred={r['parsed']['weight_class'] if r['success'] else 'FAILED'} -- {r['parsed']['reasoning'] if r['success'] else r['error']}")

[209] gt=1 pred=0 -- The gait shows symmetrical limb movement with consistent step length and proper foot clearance during each step. Arms swing naturally in coordination with the legs, and there is no evidence of dragging, hesitation, or uneven weight distribution, indicating normal walking.
[815] gt=1 pred=0 -- The gait shows symmetrical limb movement with consistent step length and proper foot clearance, and the arms swing naturally in coordination with the legs, indicating normal walking without signs of limping or shuffling.
[1710] gt=1 pred=1 -- The gait shows a clear asymmetry with the right leg taking longer, more powerful steps while the left leg appears to be dragging or moving with reduced amplitude, consistent with a limp. There is also noticeable uneven weight distribution and foot clearance, indicating one leg is compensating for the other.


In [3]:
!pip install -q -U transformers accelerate qwen-vl-utils opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 89.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 56.8 MB/s eta 0:00:00:00:0100:01


In [4]:
import sys
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')

from frame_sampling import sample_frame_indices, extract_frames
from kaggle_qwen_backend import generate_fn
from harness import classify_frames_local, parse_model_response
from labels import load_labels, get_instance_label
from skeleton_utils import load_instance_joints, active_frame_range

In [6]:
import cv2

DATA_DIR = '/kaggle/input/datasets/arnobsaha1/weightgait-pilot'
CSV_PATH = f'{DATA_DIR}/Absolute_Data.csv'
labels = load_labels(CSV_PATH)

prompt = (
    'You are an expert in clinical gait analysis. You are shown a sequence of frames '
    'sampled evenly across a video of a person walking. Carefully examine their '
    'walking movement across the frames, paying attention to: step length and '
    'symmetry between the left and right leg, foot clearance from the ground on '
    'each step, arm swing pattern, stride timing, and any dragging, hesitation, or '
    'uneven weight distribution between the legs. '
    'Base your judgment on the walking movement pattern itself, not on clothing, '
    'footwear, or other visual details unrelated to how the person moves. '
    'Classify their gait into exactly one of the following three categories: '
    '0 = normal walking, 1 = limping (one leg affected), 2 = shuffling (both legs '
    'affected). Respond only with valid JSON in this exact format: '
    '{"weight_class": <0, 1, or 2>, "reasoning": "<2-3 sentences describing the '
    'specific gait characteristics you observed that led to this classification>"}'
)

# warm-up call -- loads the model onto the GPU and confirms everything works
VIDEO_PATH = f'{DATA_DIR}/Instance_785.0.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

instance_joints = load_instance_joints(CSV_PATH, 785)
start, end = active_frame_range(instance_joints)
indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
frames = extract_frames(VIDEO_PATH, indices)

result = classify_frames_local(generate_fn, frames, prompt)
print(result)
print("Ground truth:", get_instance_label(labels, 785))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

{'video_path': None, 'raw_response': '{"weight_class": 0, "reasoning": "The person demonstrates a symmetrical gait with consistent step length and timing, normal foot clearance, and coordinated arm swing. There is no evidence of dragging, hesitation, or uneven weight distribution, indicating normal walking."}', 'parsed': {'weight_class': 0, 'reasoning': 'The person demonstrates a symmetrical gait with consistent step length and timing, normal foot clearance, and coordinated arm swing. There is no evidence of dragging, hesitation, or uneven weight distribution, indicating normal walking.'}, 'success': True, 'error': None}
Ground truth: {'weight_class': 0, 'gait_freeze': False, 'obstacle': False, 'person_id': 13}


In [7]:
SKELETON_INSTANCES = [71,189,438,494,727,785,795,851,1690,1757,
                       91,209,458,514,747,805,815,871,1710,1777,
                       111,229,478,534,767,825,835,891,1730,1797]
GT = {71:0,189:0,438:0,494:0,727:0,785:0,795:0,851:0,1690:0,1757:0,
      91:1,209:1,458:1,514:1,747:1,805:1,815:1,871:1,1710:1,1777:1,
      111:2,229:2,478:2,534:2,767:2,825:2,835:2,891:2,1730:2,1797:2}

In [9]:
from PIL import Image
import cv2
import json
import kaggle_qwen_backend as kqb
from harness import parse_model_response

def generate_greedy(frames, prompt, max_new_tokens=512):
    model, processor = kqb._load()
    images = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def classify_greedy(frames, prompt):
    raw_text = generate_greedy(frames, prompt)
    parsed, success, error = parse_model_response(raw_text)
    return {"video_path": None, "raw_response": raw_text, "parsed": parsed, "success": success, "error": error}

def classify_with_temp(frames, prompt, temperature=None):
    raw_text = generate_fn(frames, prompt, temperature=temperature)
    parsed, success, error = parse_model_response(raw_text)
    return {"video_path": None, "raw_response": raw_text, "parsed": parsed, "success": success, "error": error}

In [10]:
TEMPS_TO_TEST = [0.0, 0.3, 0.7]

temp_sweep_skeleton = {}
for inst in SKELETON_INSTANCES:
    vp = f'{DATA_DIR}/kaggle_representations/{inst}/skeleton.mp4'
    ij = load_instance_joints(CSV_PATH, inst)
    s, e = active_frame_range(ij)
    cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    idx = sample_frame_indices(tf, n=8, start=s, end=e)
    fr = extract_frames(vp, idx)

    row = {}
    for t in TEMPS_TO_TEST:
        r = classify_greedy(fr, prompt) if t == 0.0 else classify_with_temp(fr, prompt, temperature=t)
        row[t] = r['parsed']['weight_class'] if r['success'] else None
    temp_sweep_skeleton[inst] = row
    print(f"[{inst}] gt={GT[inst]} preds_by_temp={row}")

json.dump({str(k): v for k, v in temp_sweep_skeleton.items()}, open('/kaggle/working/temp_sweep_skeleton.json', 'w'))
print("\nSaved to /kaggle/working/temp_sweep_skeleton.json")

[71] gt=0 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[189] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[438] gt=0 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[494] gt=0 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[727] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[785] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[795] gt=0 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 0}
[851] gt=0 preds_by_temp={0.0: 0, 0.3: 1, 0.7: 1}
[1690] gt=0 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[1757] gt=0 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[458] gt=1 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[747] gt=1 preds_by_temp={0.0: 0, 0.3: 1, 0.7: 1}
[805] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[815] gt=1 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[871] gt=1 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[1710] gt=1 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[1777] gt=1 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[111] gt=2 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 1}
[229] gt=2 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 1}
[478] gt=2 preds_by_temp={0.0: 1, 0.3: 1, 0.7: 

In [11]:
STRATIFIED = [785, 1690, 91, 1710, 825, 1730]

temp_sweep_others = {}
for inst in STRATIFIED:
    ij = load_instance_joints(CSV_PATH, inst)
    s, e = active_frame_range(ij)
    for rep in ['raw_video', 'overlay', 'montage']:
        if rep == 'montage':
            fr = [cv2.imread(f'{DATA_DIR}/kaggle_representations/{inst}/montage.png')]
        else:
            vp = (f'{DATA_DIR}/Instance_{inst}.0.mp4' if rep == 'raw_video'
                  else f'{DATA_DIR}/kaggle_representations/{inst}/overlay.mp4')
            cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
            idx = sample_frame_indices(tf, n=8, start=s, end=e)
            fr = extract_frames(vp, idx)

        row = {}
        for t in TEMPS_TO_TEST:
            r = classify_greedy(fr, prompt) if t == 0.0 else classify_with_temp(fr, prompt, temperature=t)
            row[t] = r['parsed']['weight_class'] if r['success'] else None
        temp_sweep_others[f"{inst}_{rep}"] = row
        print(f"[{inst}][{rep}] gt={GT[inst]} preds_by_temp={row}")

json.dump(temp_sweep_others, open('/kaggle/working/temp_sweep_others.json', 'w'))
print("\nSaved to /kaggle/working/temp_sweep_others.json")

[785][raw_video] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[785][overlay] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[785][montage] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1690][raw_video] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1690][overlay] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1690][montage] gt=0 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[91][raw_video] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[91][overlay] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[91][montage] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1710][raw_video] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1710][overlay] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1710][montage] gt=1 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[825][montage] gt=2 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1730][raw_video] gt=2 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1730][overlay] gt=2 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}
[1730][montage] gt=2 preds_by_temp={0.0: 0, 0.3: 0, 0.7: 0}

Saved to /kaggle/working/temp_sweep_oth

In [12]:
import json
data = json.load(open('/kaggle/working/temp_sweep_skeleton.json'))
for inst in [91, 209, 514, 534, 767, 825, 835, 891]:
    print(f"[{inst}] gt={GT[int(inst)]} preds_by_temp={data[str(inst)]}")

[91] gt=1 preds_by_temp={'0.0': 1, '0.3': 1, '0.7': 1}
[209] gt=1 preds_by_temp={'0.0': 1, '0.3': 1, '0.7': 1}
[514] gt=1 preds_by_temp={'0.0': 0, '0.3': 1, '0.7': 1}
[534] gt=2 preds_by_temp={'0.0': 1, '0.3': 1, '0.7': 1}
[767] gt=2 preds_by_temp={'0.0': 0, '0.3': 0, '0.7': 0}
[825] gt=2 preds_by_temp={'0.0': 0, '0.3': 0, '0.7': 0}
[835] gt=2 preds_by_temp={'0.0': 0, '0.3': 0, '0.7': 0}
[891] gt=2 preds_by_temp={'0.0': 1, '0.3': 1, '0.7': 0}


In [4]:
ZERO_SHOT_PROMPT = (
    'You are shown a sequence of frames sampled evenly across a video of a person walking. '
    'Classify their gait into exactly one of the following three categories: '
    '0 = normal walking, 1 = limping (one leg affected), 2 = shuffling (both legs affected). '
    'Respond only with valid JSON in this exact format: '
    '{"weight_class": <0, 1, or 2>, "reasoning": "<one sentence explaining what you observed>"}'
)

FEW_SHOT_PROMPT = (
    'You are an expert in clinical gait analysis. Below are three worked examples of how to '
    'classify gait from a sequence of walking frames, followed by a new case for you to classify.\n\n'
    'Example 1 (Normal): The person shows symmetrical step length between left and right legs, '
    'consistent foot clearance from the ground, coordinated arm swing, and steady stride timing, '
    'with no dragging or hesitation. Classification: {"weight_class": 0, "reasoning": "Gait is '
    'symmetrical and rhythmic with no signs of impairment."}\n\n'
    'Example 2 (Limp): One leg takes shorter, more hesitant steps than the other, with reduced '
    'foot clearance and possible dragging on that side, and a visible asymmetry in stride length '
    'and timing between the two legs. Classification: {"weight_class": 1, "reasoning": "Clear '
    'asymmetry between legs with reduced clearance and dragging on one side, consistent with a '
    'unilateral limp."}\n\n'
    'Example 3 (Shuffle): Both legs show reduced step length and foot clearance, with a slow, '
    'shuffling gait pattern, minimal knee lift, and reduced heel-to-toe motion on both sides. '
    'Classification: {"weight_class": 2, "reasoning": "Both legs show reduced amplitude and '
    'clearance, consistent with a bilateral shuffling gait."}\n\n'
    'Now examine the following sequence of frames sampled evenly across a video of a new person '
    'walking, and classify their gait using the same three categories. Respond only with valid '
    'JSON in this exact format: {"weight_class": <0, 1, or 2>, "reasoning": "<2-3 sentences '
    'describing the specific gait characteristics you observed that led to this classification>"}'
)

PROMPTS_TO_TEST = {'zero_shot': ZERO_SHOT_PROMPT, 'few_shot': FEW_SHOT_PROMPT}

# safe redefinition in case this is a fresh kernel
from PIL import Image
import cv2, json
import kaggle_qwen_backend as kqb
from harness import parse_model_response

def generate_greedy(frames, p, max_new_tokens=512):
    model, processor = kqb._load()
    images = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": p})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def classify_greedy(frames, p):
    raw_text = generate_greedy(frames, p)
    parsed, success, error = parse_model_response(raw_text)
    return {"video_path": None, "raw_response": raw_text, "parsed": parsed, "success": success, "error": error}

In [5]:
prompt_sweep_skeleton = {}
for inst in SKELETON_INSTANCES:
    vp = f'{DATA_DIR}/kaggle_representations/{inst}/skeleton.mp4'
    ij = load_instance_joints(CSV_PATH, inst)
    s, e = active_frame_range(ij)
    cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    idx = sample_frame_indices(tf, n=8, start=s, end=e)
    fr = extract_frames(vp, idx)

    row = {}
    for name, p in PROMPTS_TO_TEST.items():
        r = classify_greedy(fr, p)
        row[name] = r['parsed']['weight_class'] if r['success'] else None
    prompt_sweep_skeleton[inst] = row
    print(f"[{inst}] gt={GT[inst]} preds_by_prompt={row}")

json.dump({str(k): v for k, v in prompt_sweep_skeleton.items()}, open('/kaggle/working/prompt_sweep_skeleton.json', 'w'))
print("\nSaved to /kaggle/working/prompt_sweep_skeleton.json")

[71] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[189] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[438] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[494] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[727] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[785] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[795] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[851] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[1690] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[1757] gt=0 preds_by_prompt={'zero_shot': 1, 'few_shot': 1}
[91] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[209] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[458] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[514] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[747] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 1}
[805] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[815] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 

In [6]:
STRATIFIED = [785, 1690, 91, 1710, 825, 1730]
prompt_sweep_others = {}
for inst in STRATIFIED:
    ij = load_instance_joints(CSV_PATH, inst)
    s, e = active_frame_range(ij)
    for rep in ['raw_video', 'overlay', 'montage']:
        if rep == 'montage':
            fr = [cv2.imread(f'{DATA_DIR}/kaggle_representations/{inst}/montage.png')]
        else:
            vp = (f'{DATA_DIR}/Instance_{inst}.0.mp4' if rep == 'raw_video'
                  else f'{DATA_DIR}/kaggle_representations/{inst}/overlay.mp4')
            cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
            idx = sample_frame_indices(tf, n=8, start=s, end=e)
            fr = extract_frames(vp, idx)

        row = {}
        for name, p in PROMPTS_TO_TEST.items():
            r = classify_greedy(fr, p)
            row[name] = r['parsed']['weight_class'] if r['success'] else None
        prompt_sweep_others[f"{inst}_{rep}"] = row
        print(f"[{inst}][{rep}] gt={GT[inst]} preds_by_prompt={row}")

json.dump(prompt_sweep_others, open('/kaggle/working/prompt_sweep_others.json', 'w'))
print("\nSaved to /kaggle/working/prompt_sweep_others.json")

[785][raw_video] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[785][overlay] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[785][montage] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[1690][raw_video] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[1690][overlay] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[1690][montage] gt=0 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[91][raw_video] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[91][overlay] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[91][montage] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[1710][raw_video] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[1710][overlay] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[1710][montage] gt=1 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[825][raw_video] gt=2 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[825][overlay] gt=2 preds_by_prompt={'zero_shot': 0, 'few_shot': 0}
[825][montage] gt=2 preds_by_prompt

In [2]:
!pip install -q -U transformers accelerate qwen-vl-utils opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 48.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import sys
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')

from frame_sampling import sample_frame_indices, extract_frames
from kaggle_qwen_backend import generate_fn
from harness import classify_frames_local, parse_model_response
from labels import load_labels, get_instance_label
from skeleton_utils import load_instance_joints, active_frame_range

import cv2

DATA_DIR = '/kaggle/input/datasets/arnobsaha1/weightgait-pilot'
CSV_PATH = f'{DATA_DIR}/Absolute_Data.csv'
labels = load_labels(CSV_PATH)

prompt = (
    'You are an expert in clinical gait analysis. You are shown a sequence of frames '
    'sampled evenly across a video of a person walking. Carefully examine their '
    'walking movement across the frames, paying attention to: step length and '
    'symmetry between the left and right leg, foot clearance from the ground during '
    'each step, arm swing pattern, stride timing, and any dragging, hesitation, or '
    'uneven weight distribution between the legs. '
    'Base your judgment on the walking movement pattern itself, not on clothing, '
    'footwear, or other visual details unrelated to how the person moves. '
    'Classify their gait into exactly one of the following three categories: '
    '0 = normal walking, 1 = limping (one leg affected), 2 = shuffling (both legs '
    'affected). Respond only with valid JSON in this exact format: '
    '{"weight_class": <0, 1, or 2>, "reasoning": "<2-3 sentences describing the '
    'specific gait characteristics you observed that led to this classification>"}'
)

SKELETON_INSTANCES = [71,189,438,494,727,785,795,851,1690,1757,
                       91,209,458,514,747,805,815,871,1710,1777,
                       111,229,478,534,767,825,835,891,1730,1797]
GT = {71:0,189:0,438:0,494:0,727:0,785:0,795:0,851:0,1690:0,1757:0,
      91:1,209:1,458:1,514:1,747:1,805:1,815:1,871:1,1710:1,1777:1,
      111:2,229:2,478:2,534:2,767:2,825:2,835:2,891:2,1730:2,1797:2}

# warm-up call -- loads the model onto the GPU
VIDEO_PATH = f'{DATA_DIR}/Instance_785.0.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
instance_joints = load_instance_joints(CSV_PATH, 785)
start, end = active_frame_range(instance_joints)
indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
frames = extract_frames(VIDEO_PATH, indices)
result = classify_frames_local(generate_fn, frames, prompt)
print(result)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

{'video_path': None, 'raw_response': '{\n  "weight_class": 0,\n  "reasoning": "The person exhibits a symmetrical gait with consistent step length and normal foot clearance during each step. Arms swing naturally in coordination with the legs, and there is no visible dragging, hesitation, or uneven weight distribution, indicating normal walking."\n}', 'parsed': {'weight_class': 0, 'reasoning': 'The person exhibits a symmetrical gait with consistent step length and normal foot clearance during each step. Arms swing naturally in coordination with the legs, and there is no visible dragging, hesitation, or uneven weight distribution, indicating normal walking.'}, 'success': True, 'error': None}


In [7]:
import json
from PIL import Image
import kaggle_qwen_backend as kqb

STRATIFIED = [785, 1690, 91, 1710, 825, 1730]
N_VALUES = [16, 32]

def generate_greedy(frames, p, max_new_tokens=512):
    model, processor = kqb._load()
    images = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": p})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def classify_greedy(frames, p):
    raw_text = generate_greedy(frames, p)
    parsed, success, error = parse_model_response(raw_text)
    return {"video_path": None, "raw_response": raw_text, "parsed": parsed, "success": success, "error": error}

In [8]:
N_VALUES = [16, 32]

n_sweep_skeleton = {}
for inst in SKELETON_INSTANCES:
    vp = f'{DATA_DIR}/kaggle_representations/{inst}/skeleton.mp4'
    ij = load_instance_joints(CSV_PATH, inst)
    s, e = active_frame_range(ij)
    cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()

    row = {}
    for n in N_VALUES:
        idx = sample_frame_indices(tf, n=n, start=s, end=e)
        fr = extract_frames(vp, idx)
        r = classify_greedy(fr, prompt)
        row[n] = r['parsed']['weight_class'] if r['success'] else None
    n_sweep_skeleton[inst] = row
    print(f"[{inst}] gt={GT[inst]} preds_by_N={row}")

json.dump({str(k): v for k, v in n_sweep_skeleton.items()}, open('/kaggle/working/n_sweep_skeleton.json', 'w'))
print("\nSaved to /kaggle/working/n_sweep_skeleton.json")

[71] gt=0 preds_by_N={16: 0, 32: 0}
[189] gt=0 preds_by_N={16: 1, 32: 0}
[438] gt=0 preds_by_N={16: 0, 32: 0}
[494] gt=0 preds_by_N={16: 0, 32: 0}
[727] gt=0 preds_by_N={16: 0, 32: 0}
[785] gt=0 preds_by_N={16: 0, 32: 0}
[795] gt=0 preds_by_N={16: 0, 32: 0}
[851] gt=0 preds_by_N={16: 0, 32: 0}
[1690] gt=0 preds_by_N={16: 1, 32: 1}
[1757] gt=0 preds_by_N={16: 1, 32: 1}
[91] gt=1 preds_by_N={16: 0, 32: 0}
[209] gt=1 preds_by_N={16: 1, 32: 0}
[458] gt=1 preds_by_N={16: 1, 32: 0}
[514] gt=1 preds_by_N={16: 0, 32: 0}
[747] gt=1 preds_by_N={16: 0, 32: 0}
[805] gt=1 preds_by_N={16: 0, 32: 0}
[815] gt=1 preds_by_N={16: 0, 32: 0}
[871] gt=1 preds_by_N={16: 0, 32: 0}
[1710] gt=1 preds_by_N={16: 1, 32: 0}
[1777] gt=1 preds_by_N={16: 1, 32: 1}
[111] gt=2 preds_by_N={16: 0, 32: 0}
[229] gt=2 preds_by_N={16: 0, 32: 0}
[478] gt=2 preds_by_N={16: 1, 32: 0}
[534] gt=2 preds_by_N={16: 1, 32: 1}
[767] gt=2 preds_by_N={16: 0, 32: 0}
[825] gt=2 preds_by_N={16: 0, 32: 0}
[835] gt=2 preds_by_N={16: 0, 32: 0}

In [8]:
STRATIFIED = [785, 1690, 91, 1710, 825, 1730]
n_sweep_others = {}
for inst in STRATIFIED:
    ij = load_instance_joints(CSV_PATH, inst)
    s, e = active_frame_range(ij)
    for rep in ['raw_video', 'overlay', 'montage']:
        if rep == 'montage':
            # montage is a fixed pre-tiled image -- frame count (N) doesn't apply to it,
            # it already bakes in a fixed number of tiles at render time
            continue
        vp = (f'{DATA_DIR}/Instance_{inst}.0.mp4' if rep == 'raw_video'
              else f'{DATA_DIR}/kaggle_representations/{inst}/overlay.mp4')
        cap = cv2.VideoCapture(vp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()

        row = {}
        for n in N_VALUES:
            idx = sample_frame_indices(tf, n=n, start=s, end=e)
            fr = extract_frames(vp, idx)
            r = classify_greedy(fr, prompt)
            row[n] = r['parsed']['weight_class'] if r['success'] else None
        n_sweep_others[f"{inst}_{rep}"] = row
        print(f"[{inst}][{rep}] gt={GT[inst]} preds_by_N={row}")

json.dump(n_sweep_others, open('/kaggle/working/n_sweep_others.json', 'w'))
print("\nSaved to /kaggle/working/n_sweep_others.json")

[785][raw_video] gt=0 preds_by_N={16: 0, 32: 0}
[785][overlay] gt=0 preds_by_N={16: 0, 32: 0}
[1690][raw_video] gt=0 preds_by_N={16: 0, 32: 0}
[1690][overlay] gt=0 preds_by_N={16: 0, 32: 0}
[91][raw_video] gt=1 preds_by_N={16: 0, 32: 0}
[91][overlay] gt=1 preds_by_N={16: 0, 32: 0}
[1710][raw_video] gt=1 preds_by_N={16: 0, 32: 0}
[1710][overlay] gt=1 preds_by_N={16: 0, 32: 0}
[825][raw_video] gt=2 preds_by_N={16: 0, 32: 0}
[825][overlay] gt=2 preds_by_N={16: 0, 32: 0}
[1730][raw_video] gt=2 preds_by_N={16: 0, 32: 0}
[1730][overlay] gt=2 preds_by_N={16: 0, 32: 0}

Saved to /kaggle/working/n_sweep_others.json


In [5]:
N_VALUES = [16, 32]

In [1]:
!pip install -q -U transformers accelerate qwen-vl-utils opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 66.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 23.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 38.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depe

In [2]:
import sys
sys.path.append('/kaggle/input/datasets/arnobsaha1/weightgait-pilot')

from frame_sampling import sample_frame_indices, extract_frames
from kaggle_qwen_backend import generate_fn
from harness import classify_frames_local, parse_model_response
from labels import load_labels, get_instance_label
from skeleton_utils import load_instance_joints, active_frame_range

In [5]:
import cv2, json
from PIL import Image
import kaggle_qwen_backend as kqb

DATA_DIR = '/kaggle/input/datasets/arnobsaha1/weightgait-pilot'
CSV_PATH = f'{DATA_DIR}/Absolute_Data.csv'
labels = load_labels(CSV_PATH)

prompt = (
    'You are an expert in clinical gait analysis. You are shown a sequence of frames '
    'sampled evenly across a video of a person walking. Carefully examine their '
    'walking movement across the frames, paying attention to: step length and '
    'symmetry between the left and right leg, foot clearance from the ground during '
    'each step, arm swing pattern, stride timing, and any dragging, hesitation, or '
    'uneven weight distribution between the legs. '
    'Base your judgment on the walking movement pattern itself, not on clothing, '
    'footwear, or other visual details unrelated to how the person moves. '
    'Classify their gait into exactly one of the following three categories: '
    '0 = normal walking, 1 = limping (one leg affected), 2 = shuffling (both legs '
    'affected). Respond only with valid JSON in this exact format: '
    '{"weight_class": <0, 1, or 2>, "reasoning": "<2-3 sentences describing the '
    'specific gait characteristics you observed that led to this classification>"}'
)

VIDEO_PATH = f'{DATA_DIR}/Instance_785.0.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'Could not open {VIDEO_PATH}'
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

instance_joints = load_instance_joints(CSV_PATH, 785)
start, end = active_frame_range(instance_joints)
indices = sample_frame_indices(total_frames, n=8, start=start, end=end)
frames = extract_frames(VIDEO_PATH, indices)
print(classify_frames_local(generate_fn, frames, prompt))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (5 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

{'video_path': None, 'raw_response': '{\n  "weight_class": 0,\n  "reasoning": "The individual exhibits a symmetrical gait with consistent step length and regular stride timing. There is no visible dragging, hesitation, or uneven weight distribution; the foot clearance and arm swing are natural and coordinated throughout the sequence."\n}', 'parsed': {'weight_class': 0, 'reasoning': 'The individual exhibits a symmetrical gait with consistent step length and regular stride timing. There is no visible dragging, hesitation, or uneven weight distribution; the foot clearance and arm swing are natural and coordinated throughout the sequence.'}, 'success': True, 'error': None}


In [6]:
def generate_greedy(frames, p, max_new_tokens=512):
    model, processor = kqb._load()
    images = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": p})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def classify_greedy(frames, p):
    raw = generate_greedy(frames, p)
    parsed, success, error = parse_model_response(raw)
    return {"video_path": None, "raw_response": raw, "parsed": parsed, "success": success, "error": error}

def classify_with_temp(frames, p, temperature=None):
    raw = generate_fn(frames, p, temperature=temperature)
    parsed, success, error = parse_model_response(raw)
    return {"video_path": None, "raw_response": raw, "parsed": parsed, "success": success, "error": error}

In [7]:
import branch_d_elicit
branch_d_elicit.run(generate_greedy, input_dir="/kaggle/input/datasets/arnobsaha1/weightgait-pilot/kaggle_representations")

Branch D elicitation: 0 done, 30 to go (3 calls each)
[1/30][71] rated 16/27 (122.5s)
[2/30][91] rated 27/27 (123.0s)
[3/30][111] rated 27/27 (122.9s)
[4/30][189] rated 27/27 (123.9s)
[5/30][209] rated 27/27 (125.2s)
[6/30][229] rated 16/27 (122.4s)
[7/30][438] rated 16/27 (122.3s)
[8/30][458] rated 16/27 (122.0s)
[9/30][478] rated 16/27 (122.5s)
[10/30][494] rated 16/27 (122.7s)
[11/30][514] rated 27/27 (124.0s)
[12/30][534] rated 8/27 (121.8s)
[13/30][727] rated 27/27 (123.5s)
[16/30][785] rated 16/27 (122.5s)
[17/30][795] rated 27/27 (123.6s)
[18/30][805] rated 27/27 (123.9s)
[19/30][815] rated 27/27 (124.1s)
[20/30][825] rated 27/27 (123.0s)
[21/30][835] rated 27/27 (122.8s)
[22/30][851] rated 16/27 (122.5s)
[23/30][871] rated 16/27 (122.2s)
[24/30][891] rated 16/27 (122.7s)
[25/30][1690] rated 8/27 (121.4s)
[26/30][1710] rated 16/27 (122.0s)
[27/30][1730] rated 16/27 (122.1s)
[28/30][1757] rated 16/27 (122.5s)
[29/30][1777] rated 27/27 (123.4s)
[30/30][1797] rated 16/27 (121.8s)


[{'instance': 71,
  'weight_class': 0,
  'ratings': {'knee_flexion_rom_left': 1,
   'knee_flexion_rom_right': 1,
   'hip_flexion_rom_left': 1,
   'hip_flexion_rom_right': 1,
   'knee_angle_asymmetry': 0,
   'hip_angle_asymmetry': 0,
   'step_length': 1,
   'step_width': 1,
   'foot_vertical_rom_left': None,
   'foot_vertical_rom_right': None,
   'foot_clearance_asymmetry': None,
   'stance_fraction_left': None,
   'stance_fraction_right': None,
   'stance_swing_asymmetry': None,
   'peak_knee_flexion_timing_left': None,
   'peak_knee_flexion_timing_right': None,
   'stride_time': None,
   'step_length_variability': None,
   'knee_angle_variability': None,
   'trunk_lean_rom': 1,
   'arm_swing_amplitude_left': 1,
   'arm_swing_amplitude_right': 1,
   'arm_swing_asymmetry': 0,
   'elbow_flexion_rom': 1,
   'elbow_height_difference': 0,
   'shoulder_height_asymmetry': 0,
   'shoulder_rotation_rom': 1},
  'status': {'knee_flexion_rom_left': 'rated',
   'knee_flexion_rom_right': 'rated',
  